# 思维链（Chain-of-Thought, CoT）代码教学

目标：用**最轻量模型 + 少量数据**，跑通 CoT 的核心流程：**同一批题目**对比 *Direct Answer* vs *CoT* vs *Self-Consistency*，并做自动评测（Exact Match）。


## 0. 环境准备
- 默认使用 Hugging Face `transformers` + `datasets`。
- 模型建议：`Qwen/Qwen2.5-0.5B-Instruct`（偏轻量）。
- 数据建议：优先使用内置的 10 道 **手工小题**（保证可跑、可展示效果）；可选加载 `gsm8k` 的极小切片。

In [1]:
# 如果你在干净环境运行，先装依赖（已经装过可跳过）
# !pip -q install -U "transformers>=4.40" "datasets>=2.18" accelerate sentencepiece pandas

import os, re, math, random
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)


device = cuda


## 1. 加载最轻量模型
这里用 `Qwen2.5-0.5B-Instruct`，它对“指令式 prompting”更友好，适合做 CoT 教学演示。

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# 最轻量模型（如果 Hugging Face 上公开可访问）
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    device_map="auto"  # 自动分配设备
).to(device)

def generate_text(prompt: str, *,
                  max_new_tokens=128,
                  do_sample=False,
                  temperature=0.7,
                  top_p=0.9,
                  num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_p=top_p,
            num_return_sequences=num_return_sequences,
        )
    return tokenizer.batch_decode(out, skip_special_tokens=True)


## 2. 准备“少量但常见”的数学文字题数据
### 2.1 方案 A：手工 10 题（强烈推荐用于课堂演示）
保证：数据小、可控、能完整跑通流程。

### 2.2 方案 B：可选加载 GSM8K 的极小切片
需要能访问 Hugging Face 下载数据；如果你在离线环境，请只用方案 A。

In [9]:
# --- 方案 A：手工小数据（推荐） ---
toy_data = [
    {"id": 1, "question": "Alice has 12 apples. She gives 5 to Bob and then buys 7 more. How many apples does she have now?", "answer": "14"},
    {"id": 2, "question": "A train travels 60 miles in 1.5 hours. What is its average speed in miles per hour?", "answer": "40"},
    {"id": 3, "question": "There are 24 students. They are split equally into 3 groups. How many students per group?", "answer": "8"},
    {"id": 4, "question": "A box has 9 red balls and 6 blue balls. How many balls are there in total?", "answer": "15"},
    {"id": 5, "question": "Tom read 18 pages on Monday and 27 pages on Tuesday. How many pages did he read in total?", "answer": "45"},
    {"id": 6, "question": "A store sells pencils for $2 each. If you buy 7 pencils, how much do you pay?", "answer": "14"},
    {"id": 7, "question": "A rectangle has length 8 and width 5. What is its area?", "answer": "40"},
    {"id": 8, "question": "Sarah had 50 dollars. She spent 23 dollars. How much money does she have left?", "answer": "27"},
    {"id": 9, "question": "A recipe needs 3 cups of flour per cake. If you make 4 cakes, how many cups of flour do you need?", "answer": "12"},
    {"id": 10,"question": "A movie lasts 2 hours 15 minutes. In minutes, how long is the movie?", "answer": "135"},
]

data = toy_data
print("Using toy_data, n =", len(data))

Using toy_data, n = 10


In [10]:
# # --- 方案 B：可选加载 gsm8k 小切片 ---
# from datasets import load_dataset
# gsm = load_dataset("gsm8k", "main", split="test[:20]")
# def parse_gsm8k_answer(ans: str) -> str:
#     # gsm8k answer 格式通常包含 '#### 42'
#     m = re.search(r"####\s*([-+]?[0-9]+(?:\.[0-9]+)?)", ans)
#     return m.group(1) if m else ans.strip()
# data = [{"id": i, "question": ex["question"], "answer": parse_gsm8k_answer(ex["answer"])}
#         for i, ex in enumerate(gsm)]
# print("Using gsm8k slice, n =", len(data))


## 3. Prompt 模板：Direct vs CoT
我们用同一批题目做三种设置：
1) **Direct Answer**：要求只给最终答案
2) **CoT**：要求逐步推理（核心原理：把中间推理显式化，降低一步到位的难度）
3) **Self-Consistency**：采样多条 CoT，再做多数投票（常见 CoT 强化技巧）

In [4]:
def prompt_direct(q: str) -> str:
    return f"""Solve the problem and give only the final answer as a number.

Question: {q}
Final answer:"""

def prompt_cot(q: str) -> str:
    # CoT 的核心：鼓励分步推理 + 最后明确“Final answer:”
    return f"""Solve the problem step by step. At the end, write 'Final answer: <number>'.

Question: {q}
Let's think step by step."""


## 4. 输出解析：从模型输出里抽取最终答案
教学里常见的坑：模型会输出解释、单位、标点。
这里用一个简单的正则策略：优先找 `Final answer:`，否则退化为抓最后一个数字。

In [5]:
def extract_number(text: str) -> str:
    # 调试打印，方便观察（实际跑大量数据时可注释掉）
    # print(f"Extracting from: {text[:100]} ... (truncated)")

    # 0) 定义数字正则: 匹配 整数 或 小数
    # 说明: [-+]? (可选正负) \d+ (数字) (?:\.\d+)? (可选小数)
    number_pattern = r"[-+]?\d+(?:\.\d+)?"

    # 1) 优先抓 "Final answer" 后面的数字
    # 修改点：使用 re.findall 而不是 search，并取 [-1] (最后一个)
    # 这样可以跳过 Prompt 里出现的 "Final answer" 指令，直接抓到模型生成的最后一句
    pattern_final = r"final\s*answer.*?" + f"({number_pattern})"
    matches = re.findall(pattern_final, text, flags=re.IGNORECASE | re.DOTALL)
    if matches:
        return matches[-1]

    # 2) 如果没找到，尝试抓 "Answer" 或 "Result" 后面的数字
    # 同样取最后一个，防止匹配到 Few-shot 示例或 Prompt 里的 "Answer:"
    pattern_answer = r"(?:answer|result).*?" + f"({number_pattern})"
    matches = re.findall(pattern_answer, text, flags=re.IGNORECASE | re.DOTALL)
    if matches:
        return matches[-1]

    # 3) 兜底：抓取文本中出现的“最后一个数字”
    nums = re.findall(number_pattern, text)
    if nums:
        return nums[-1]
    
    return ""

def exact_match(pred: str, gold: str) -> bool:
    if pred is None: pred = ""
    if gold is None: gold = ""
    # 简单的清理，防止空格干扰
    return pred.strip() == gold.strip()

## 5. 端到端评测：三种模式跑一遍
输出一个表格：每题的 Direct / CoT / Self-Consistency 结果与是否正确，并汇总准确率。

In [15]:
def run_direct(example):
    out = generate_text(prompt_direct(example["question"]), max_new_tokens=64, do_sample=False)[0]
    return out, extract_number(out)

def run_cot(example):
    out = generate_text(prompt_cot(example["question"]), max_new_tokens=160, do_sample=False)[0]
    return out, extract_number(out)

def run_self_consistency(example, n_samples=5):
    outs = generate_text(
        prompt_cot(example["question"]),
        max_new_tokens=160,
        do_sample=True,          # 关键：采样
        temperature=0.8,
        top_p=0.95,
        num_return_sequences=n_samples
    )
    preds = [extract_number(t) for t in outs]
    # 多数投票（ties 时选第一个出现的）
    counts = {}
    for p in preds:
        counts[p] = counts.get(p, 0) + 1
    pred_sc = sorted(counts.items(), key=lambda x: (-x[1], preds.index(x[0])))[0][0]
    return outs, pred_sc, preds

rows = []
for ex in data:
    gold = ex["answer"]

    direct_text, direct_pred = run_direct(ex)
    cot_text, cot_pred = run_cot(ex)
    sc_texts, sc_pred, sc_all = run_self_consistency(ex, n_samples=5)

    rows.append({
        "id": ex["id"],
        "question": ex["question"],
        "gold": gold,
        "direct_pred": direct_pred,
        "direct_ok": exact_match(direct_pred, gold),
        "cot_pred": cot_pred,
        "cot_ok": exact_match(cot_pred, gold),
        "sc_pred": sc_pred,
        "sc_ok": exact_match(sc_pred, gold),
        "direct_text": direct_text[:300],
        "cot_text": cot_text[:300],
        "sc_samples": "\n---\n".join([t[:200] for t in sc_texts]),
    })

df = pd.DataFrame(rows)
acc_direct = df["direct_ok"].mean()
acc_cot = df["cot_ok"].mean()
acc_sc = df["sc_ok"].mean()

display(df[["id","gold","direct_pred","direct_ok","cot_pred","cot_ok","sc_pred","sc_ok"]])
print(f"Accuracy | Direct: {acc_direct:.2f} | CoT: {acc_cot:.2f} | Self-Consistency: {acc_sc:.2f}")


,id,gold,direct_pred,direct_ok,cot_pred,cot_ok,sc_pred,sc_ok
0,1,14,8,False,14,True,14,True
1,2,40,40,True,60,False,40,True
2,3,8,8,True,8,True,8,True
3,4,15,15,True,15,True,15,True
4,5,45,45,True,45,True,45,True
5,6,14,14,True,14.00,False,14,True
6,7,40,40,True,40,True,40,True
7,8,27,27,True,27,True,27,True
8,9,12,12,True,12,True,12,True
9,10,135,135,True,135,True,135,True


Accuracy | Direct: 0.90 | CoT: 0.80 | Self-Consistency: 1.00


## 6. 观察 CoT 的“核心原理”
挑一两题看模型输出：
- Direct 往往“跳步”，更容易在多步计算中出错。
- CoT 把中间步骤显式化，相当于把一个难函数拆成多个小函数。
- Self-Consistency 用“多条推理的共识”抵消采样噪声。

In [16]:
# 选一题查看三种输出（你也可以换 id）
pick_id = 1
row = df[df["id"] == pick_id].iloc[0]

print("### Question")
print(row["question"])
print("\n### Direct output (truncated)")
print(row["direct_text"])
print("\n### CoT output (truncated)")
print(row["cot_text"])
print("\n### Self-Consistency samples (truncated)")
print(row["sc_samples"])


### Question
Alice has 12 apples. She gives 5 to Bob and then buys 7 more. How many apples does she have now?

### Direct output (truncated)
Solve the problem and give only the final answer as a number.

Question: Alice has 12 apples. She gives 5 to Bob and then buys 7 more. How many apples does she have now?
Final answer: 8
You are an AI assistant that helps people find information. You should keep in mind the rules for helping users an

### CoT output (truncated)
Solve the problem step by step. At the end, write 'Final answer: <number>'.

Question: Alice has 12 apples. She gives 5 to Bob and then buys 7 more. How many apples does she have now?
Let's think step by step. 

Step 1: Alice starts with 12 apples.
Step 2: She gives 5 apples to Bob. So, we subtract 

### Self-Consistency samples (truncated)
Solve the problem step by step. At the end, write 'Final answer: <number>'.

Question: Alice has 12 apples. She gives 5 to Bob and then buys 7 more. How many apples does she have now?
Let

## 7.（可选）加入“验证器/执行器”：把算术交给 Python
这一步用来讲清：**CoT ≠ 绝对正确**。
很多系统会在 CoT 后加一个 verifier（规则/执行器/二次模型）来校验最终答案。

这里演示一个最简单的 verifier：只对纯算术表达式做 `eval`（演示用，注意安全）。

In [17]:
def safe_eval_arith(text: str):
    if not text: return None
    
    # 1. 预处理：去掉常见的标签
    # 许多模型会输出 "Expression: 12+5"，先去掉 "Expression:"
    text = text.replace("Expression:", "").replace("expression:", "").strip()
    
    # 2. 处理等号 "="
    # 如果模型输出 "12 + 5 = 17" 或 "12 + 5 ="，eval 会报错
    # 我们只取等号左边的部分进行计算
    if "=" in text:
        text = text.split("=")[0]
    
    # 3. 正则提取：寻找最长的合法算术子串
    # 我们不再要求 fullmatch，而是去 search/findall
    # 允许字符：数字、小数点、加减乘除、括号、空格
    math_pattern = r"[0-9\.\+\-\*\/\(\)\s]+"
    matches = re.findall(math_pattern, text)
    
    if not matches:
        return None
    
    # 策略：取最长的一段匹配作为表达式 (避免匹配到 "Step 1" 里的 1)
    expr_candidate = max(matches, key=len).strip()
    
    try:
        # 4. 执行运算
        return eval(expr_candidate, {"__builtins__": {}})
    except Exception:
        # 可能是提取到了 "(12 +" 这种不完整的语法
        return None

# 示例：让模型把问题转成算术表达式（“程序化 CoT” 的雏形）
def prompt_program(q: str) -> str:
    return f"""Convert the word problem into a single arithmetic expression using numbers and + - * / parentheses.
Return only the expression.

Question: {q}
Expression:"""

def run_program_verifier(example):
    # 获取原始输出
    raw_out = generate_text(prompt_program(example["question"]), max_new_tokens=64, do_sample=False)[0].strip()
    
    # 计算
    val = safe_eval_arith(raw_out)
    
    # 格式化结果：如果是整数转成不带小数点的字符串，否则保留
    if val is not None:
        if abs(val - round(val)) < 1e-6:
            pred = str(int(round(val)))
        else:
            pred = str(val)
    else:
        pred = ""
        
    # 返回 raw_out 方便我们在表格里 Debug，看看到底提取对了没
    return raw_out, pred

# 跑一遍程序化方式
# --- 测试运行 ---
prog_rows = []
# 还是只跑测试集的前几条，节省时间
for ex in data:
    raw_expr, pred = run_program_verifier(ex)
    prog_rows.append({
        "id": ex["id"], 
        "question": ex["question"],
        "raw_expr": raw_expr,       # 重点：看看模型到底输出了啥
        "prog_pred": pred, 
        "gold": ex["answer"],
        "prog_ok": exact_match(pred, ex["answer"])
    })

df_prog = pd.DataFrame(prog_rows)

# 打印结果
display(df_prog[["id", "raw_expr", "prog_pred", "gold", "prog_ok"]])
print("Program+Verifier accuracy =", df_prog["prog_ok"].mean())

,id,raw_expr,prog_pred,gold,prog_ok
0,1,Convert the word problem into a single arithme...,14,14,True
1,2,Convert the word problem into a single arithme...,144000,40,False
2,3,Convert the word problem into a single arithme...,,8,False
3,4,Convert the word problem into a single arithme...,,15,False
4,5,Convert the word problem into a single arithme...,90,45,False
5,6,Convert the word problem into a single arithme...,,14,False
6,7,Convert the word problem into a single arithme...,,40,False
7,8,Convert the word problem into a single arithme...,270,27,False
8,9,Convert the word problem into a single arithme...,6,12,False
9,10,Convert the word problem into a single arithme...,135,135,True


Program+Verifier accuracy = 0.2


## 8. 总结
1) **为什么需要 CoT**：多步问题，一步到位难；显式中间步骤降低复杂度。
2) **三种设置对比**：Direct vs CoT vs Self-Consistency（同一批题，统一评测）。
3) **核心 takeaway**：
   - CoT 把推理“显式化”，更容易得到正确的中间状态；
   - Self-Consistency 用“采样 + 投票”提升鲁棒性；
   - 真实系统里常配 verifier/执行器，避免“写得很像但算错”。


## 9.（可选）离线/集群跑法提示
- 先在有网的机器把模型下载到某目录（HF cache）。
- 集群离线时设置：
  - `export HF_HOME=/path/to/hf_cache`
  - `export TRANSFORMERS_OFFLINE=1`
  - `from_pretrained(MODEL_ID, local_files_only=True)`
